 **Abstractive Text Summarization using BART**

*   BART - Bidirectional and Auto Regressie Transformer(BERT + GPT)



In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import pandas as pd

In [ ]:
!pip install "transformers<5.0.0"

In [ ]:
!pip install rouge-score bert-score nltk
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
!pip install datasets

In [ ]:
#Loading the dataset
from datasets import load_dataset
ds = load_dataset("knkarthick/dialogsum")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [ ]:
ds['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [ ]:
ds['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

1. WITHOUT FINE - TUNING


In [ ]:
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cuda:0


In [ ]:
article_1 = ds['train'][1]['dialogue']

In [ ]:
pipe(article_1, max_length= 20, min_length= 10, do_sample= False)

[{'summary_text': 'Ricky has received his Polio, Tetanus and Hepatitis B shots.'}]




2.   With Fine Tuning



In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
#Tokenization
def preprocess_function(batch):
    source = batch['dialogue']
    target = batch['summary']
    source_ids = tokenizer(source, truncation= True, padding= 'max_length', max_length= 128)
    target_ids = tokenizer(target, truncation= True, padding= 'max_length', max_length= 128)

    #labels for loss computation
    labels = target_ids['input_ids']
    labels = [[(label if label != tokenizer.pad_token_id else -100 )
                for label in labels_example]
                for labels_example in labels]

    return{
        "input_ids": source_ids['input_ids'],
        "attention_mask": source_ids['attention_mask'],
        "labels": labels
    }

In [ ]:
df_source = ds.map(preprocess_function, batched= True )

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
#training arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir= 'bart_model',
    per_device_train_batch_size= 8,
    num_train_epochs = 2,
    remove_unused_columns= True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = df_source['train'], eval_dataset = df_source['test']
)

In [ ]:
trainer.train()

Step,Training Loss
500,1.592100
1000,1.488800
1500,1.433500
2000,1.083700
2500,1.018200
3000,1.000500


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3116, training_loss=1.260661579372029, metrics={'train_runtime': 4281.9426, 'train_samples_per_second': 5.82, 'train_steps_per_second': 0.728, 'total_flos': 6750530835578880.0, 'train_loss': 1.260661579372029, 'epoch': 2.0})

In [ ]:
eval_results = trainer.evaluate()

In [ ]:
eval_results

{'eval_loss': 1.6681321859359741,
 'eval_runtime': 56.9374,
 'eval_samples_per_second': 26.345,
 'eval_steps_per_second': 3.302,
 'epoch': 2.0}

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
model.generation_config.early_stopping = True
model.generation_config.num_beams = 4
model.generation_config.length_penalty = 2.0

#Saving The Model
model.save_pretrained('/content/drive/MyDrive/bart_nlp')
tokenizer.save_pretrained('/content/drive/MyDrive/bart_nlp')

tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/bart_nlp')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/drive/MyDrive/bart_nlp')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def summarize(blog_post):
    inputs = tokenizer(
        blog_post,
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    summary_ids = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=150,
        min_length=10,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
        forced_bos_token_id=tokenizer.bos_token_id
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

#evaluate_summary() function
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_summary(generated_summary, reference_summary):
    # BLEU
    reference_tokens = [reference_summary.split()]
    generated_tokens = generated_summary.split()
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smoothie)

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(reference_summary, generated_summary)

    # BERTScore
    P, R, F1 = bert_score([generated_summary], [reference_summary], lang='en', verbose=False)

    print("=" * 50)
    print(f"BLEU Score       : {bleu:.4f}")
    print("-" * 50)
    print(f"ROUGE-1  → P: {rouge_scores['rouge1'].precision:.4f} | R: {rouge_scores['rouge1'].recall:.4f} | F1: {rouge_scores['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2  → P: {rouge_scores['rouge2'].precision:.4f} | R: {rouge_scores['rouge2'].recall:.4f} | F1: {rouge_scores['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L  → P: {rouge_scores['rougeL'].precision:.4f} | R: {rouge_scores['rougeL'].recall:.4f} | F1: {rouge_scores['rougeL'].fmeasure:.4f}")
    print("-" * 50)
    print(f"BERTScore → P: {P.mean():.4f} | R: {R.mean():.4f} | F1: {F1.mean():.4f}")
    print("=" * 50)

    return {
        "bleu": bleu,
        "rouge1": rouge_scores['rouge1'],
        "rouge2": rouge_scores['rouge2'],
        "rougeL": rouge_scores['rougeL'],
        "bert_F1": F1.mean().item()
    }

#WITHOUT Fine-Tuning evaluation
article_1 = ds['train'][1]['dialogue']
reference_1 = ds['train'][1]['summary']

no_ft_summary = pipe(article_1, max_length=20, min_length=10, do_sample=False)[0]['summary_text']

print("\n=== WITHOUT Fine-Tuning ===")
print(f"Generated : {no_ft_summary}")
print(f"Reference : {reference_1}")
metrics_no_ft = evaluate_summary(no_ft_summary, reference_1)

#WITH Fine-Tuning evaluation
ft_summary = summarize(article_1)

print(f"\n[DEBUG] Fine-tuned raw output: '{ft_summary}'")
if not ft_summary.strip():
    print("[WARNING] Model generated empty output. Check device placement and generation params.")

print("\n=== WITH Fine-Tuning ===")
print(f"Generated : {ft_summary}")
print(f"Reference : {reference_1}")
if ft_summary.strip():
    metrics_ft = evaluate_summary(ft_summary, reference_1)
else:
    print("Skipping evaluation — generated summary is empty.")

# ── Side-by-side Comparison
comparison = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "BERTScore F1"],
    "Without Fine-Tuning": [
        metrics_no_ft["bleu"],
        metrics_no_ft["rouge1"].fmeasure,
        metrics_no_ft["rouge2"].fmeasure,
        metrics_no_ft["rougeL"].fmeasure,
        metrics_no_ft["bert_F1"]
    ],
    "With Fine-Tuning": [
        metrics_ft["bleu"],
        metrics_ft["rouge1"].fmeasure,
        metrics_ft["rouge2"].fmeasure,
        metrics_ft["rougeL"].fmeasure,
        metrics_ft["bert_F1"]
    ]
})

comparison["Improvement"] = (comparison["With Fine-Tuning"] - comparison["Without Fine-Tuning"]).round(4)
print("\n", comparison.to_string(index=False))

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(



=== WITHOUT Fine-Tuning ===
Generated : Ricky has received his Polio, Tetanus and Hepatitis B shots.
Reference : Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU Score       : 0.0139
--------------------------------------------------
ROUGE-1  → P: 0.3000 | R: 0.1667 | F1: 0.2143
ROUGE-2  → P: 0.0000 | R: 0.0000 | F1: 0.0000
ROUGE-L  → P: 0.3000 | R: 0.1667 | F1: 0.2143
--------------------------------------------------
BERTScore → P: 0.8440 | R: 0.8716 | F1: 0.8576


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(



[DEBUG] Fine-tuned raw output: 'Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for hepatitis A, Chickenpox and Measles shots.'

=== WITH Fine-Tuning ===
Generated : Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for hepatitis A, Chickenpox and Measles shots.
Reference : Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BLEU Score       : 0.0148
--------------------------------------------------
ROUGE-1  → P: 0.1923 | R: 0.2778 | F1: 0.2273
ROUGE-2  → P: 0.0000 | R: 0.0000 | F1: 0.0000
ROUGE-L  → P: 0.1538 | R: 0.2222 | F1: 0.1818
--------------------------------------------------
BERTScore → P: 0.8285 | R: 0.8727 | F1: 0.8501

       Metric  Without Fine-Tuning  With Fine-Tuning  Improvement
        BLEU             0.013872          0.014817       0.0009
  ROUGE-1 F1             0.214286          0.227273       0.0130
  ROUGE-2 F1             0.000000          0.000000       0.0000
  ROUGE-L F1             0.214286          0.181818      -0.0325
BERTScore F1             0.857604          0.850052      -0.0076
